# Week 12 — AI Applications in Raster and 3D Data

**Spatial Data Science with Python · MSc GIS / AGMS 1333**  
**Official Table 4 topic:** Artificial Intelligence · Raster masks, segmentation, IoU

**Learning outcome.** Measure mask agreement on held-out image columns.

**What you will implement:** Binary arrays, threshold selection, IoU.  
**Mathematical model:** IoU=TP/(TP+FP+FN).

> **Data provenance:** The Urban Teaching Grid is a **synthetic** 12 × 12 grid at 250-m cell spacing. Its temperatures and facilities are simulated; do not present its numerical outputs as field measurements. All distance calculations in the labs use the projected coordinates EPSG:32647 where relevant.

**How to run.** In Colab, upload this notebook from *File → Upload notebook*. Run each cell top-to-bottom. The setup cell below will ask for `DATA_ONLY.zip` only if the data folder is absent. In JupyterLab, extract the complete ZIP and open this notebook in `01_Core_Labs/`.


## 1. Prepare the environment and locate the data

The same setup works in JupyterLab and Google Colab. It selects an existing local data directory, or prompts for the packaged dataset in Colab. `assert` fails early if the archive is incomplete.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## 2. Key concepts and expected data

The analysis follows **input → cleaning → computation → validation → interpretation**. Always inspect observation keys, coordinate units, missing values, and the intended study period before fitting a model.

**Mathematical foundation:** IoU=TP/(TP+FP+FN)

**Method family:** Binary arrays, threshold selection, IoU.

## 3. Main computation

Run this cell after the setup. The block calculates the principal statistics or fits the weekly model. The original course code is retained so results remain aligned with the slide examples.

In [ ]:
rng=np.random.default_rng(2026)
r,c=np.mgrid[:80,:120]
reference=(((r-28)**2/21**2+(c-33)**2/24**2)<1)|(((r-52)**2/15**2+(c-89)**2/23**2)<1)
score=np.clip(reference*.73+(~reference)*.24+rng.normal(0,.2,reference.shape),0,1)
def iou(a,b):return np.count_nonzero(a&b)/max(1,np.count_nonzero(a|b))
train_ref=reference[:,:80];train_score=score[:,:80]
threshold=max(np.linspace(.2,.8,61),key=lambda t:iou(train_ref,train_score>=t))
print('Threshold selected on western columns:',round(float(threshold),2))

### Reading the main code

**Reference and prediction.** Synthetic masks represent positive versus background pixels, and a noisy score surface mimics model confidence.

- `np.count_nonzero` — Counts True Boolean conditions for confusion-matrix or IoU calculations.

**Verification questions:** Which variables contain the inputs? What new object does the cell create? What numerical result should be checked before continuing?

## 4. Map, plot, or evaluate the result

The final block turns results into a labelled graphic or output artifact. Keep units in axes/legends, and distinguish model predictions from observations.

In [ ]:
actual=reference[:,80:]; predicted=(score>=threshold)[:,80:]
tp=np.count_nonzero(actual&predicted);fp=np.count_nonzero(~actual&predicted);fn=np.count_nonzero(actual&~predicted);tn=np.count_nonzero(~actual&~predicted)
print('TP, FP, FN, TN:',tp,fp,fn,tn)
print('IoU:',iou(actual,predicted),'Pixel accuracy:',(tp+tn)/(tp+fp+fn+tn))
fig,axs=plt.subplots(1,2,figsize=(9,4));axs[0].imshow(actual,cmap='Greens');axs[0].set_title('East reference');axs[1].imshow(predicted,cmap='Greens');axs[1].set_title('East predicted');plt.show()

### Reading the output code

**Threshold selection.** Choose the cutoff on west pixels only, then apply it unchanged to east pixels.

- `np.count_nonzero` — Counts True Boolean conditions for confusion-matrix or IoU calculations.
- `plt.` — Creates a figure; title, axes, and legend are part of the analytical interpretation.

**IoU.** Count TP/FP/FN for the foreground. Pixel accuracy may be high because the background is common, even when foreground overlap is imperfect.

## 5. Critical interpretation

### Interpret your results

Complete the following short report **after** running the notebook:

1. **Research question:** What geographic question does this notebook address?
2. **Observation unit and period:** What does one row/pixel represent? What is its date and CRS?
3. **Method:** Why is this method suitable, and what assumptions does it make?
4. **Results:** Report two numerical outputs or figure patterns with units.
5. **Validity:** State one data limitation and one model/evaluation limitation.
6. **Extension:** Modify one parameter, compare results, and describe what changed.


## 6. Questions and independent practice

1. Compare IoU with pixel accuracy for two class imbalances; explain which metric is relevant to the task.
2. What would happen if the spatial unit, study date, or CRS changed?
3. Is this analysis descriptive, inferential, predictive, or a decision-support example? Justify your answer.
4. Which assumption should be tested before use in a real project?

**Next step:** Open the matching advanced notebook in `02_Advanced_Experiments/` for four short method variations.